# Proyecto capstone: predicción de ROP (Rate of Penetration)

Análisis del dataset `Well_58-32_processed_pason_log.csv` (pozo de utah, datos ya procesados).

**Objetivo:** entender cómo se comporta cada variable de perforación respecto al ROP, aplicando lo visto en el curso, para más adelante poder predecir el ROP en un pozo nuevo.

**Flujo del análisis:** carga → exploración → limpieza de columnas → correlación (con fundamento teórico de cada variable) → selección final de columnas → distribución → outliers → tramos de profundidad → estadística descriptiva → escalado (preparación para un futuro modelo).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Importar librerías y cargar el dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import RobustScaler

In [ ]:
df = pd.read_csv('Well_58-32_processed_pason_log.csv')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Well_58-32_processed_pason_log.csv'

### Aclaración: ¿qué significa "ROP(1 ft)"?

ROP (**Rate of Penetration**) es una velocidad: distancia perforada por unidad de tiempo (pies por hora, ft/hr). El `(1 ft)` del nombre original de la columna **no es la unidad** — indica que el valor se calculó tomando como referencia cada 1 pie de profundidad perforada (la resolución de muestreo), no que el ROP se mida en pies.

Para que no genere confusión más adelante, renombramos las columnas para que el nombre refleje la unidad real.

In [ ]:
df = df.rename(columns={'ROP(1 ft)': 'ROP (ft/hr)', 'ROP(1 m)': 'ROP (m/hr)'})
df.columns

## 2. Exploración inicial: forma, tipos de datos y nulos

In [ ]:
print('Shape:', df.shape)
df.dtypes

In [ ]:
df.isna().sum()

### Nota importante: ¿por qué no usamos Encoding acá?

En las notebooks del curso usamos `LabelEncoder`, `get_dummies` u `OneHotEncoder` para convertir columnas **categóricas** (texto) en números. Acá **no hace falta ningún encoder**, porque `df.dtypes` muestra que las 27 columnas ya son `float64` — todo numérico desde el origen (son mediciones de sensores, no texto).

**Lo que sí estaría bueno tener, y no tenemos:** una columna de **litología** (tipo de roca en cada profundidad: arenisca, arcilla, caliza, etc.). Esa sí sería una variable categórica, y ahí un `OneHotEncoder` o `get_dummies` tendría sentido real — nos permitiría separar el análisis por tipo de roca en vez de mezclarlo todo por profundidad, como venimos discutiendo. Si en algún momento conseguimos ese dato, esta es la sección donde se agregaría el encoding.

## 3. Limpieza: columnas duplicadas por unidad (imperial vs métrico)

El dataset trae cada variable dos veces, en dos sistemas de unidades. Nos quedamos con las columnas en **unidades imperiales** (ft, psi, k-lbs, gal/min), porque son las que se usan en la práctica en la industria de perforación (incluido en Argentina), y descartamos las métricas duplicadas.

Trabajamos sobre una copia nueva (`df_imperial`), sin tocar `df` (el dataset original tal cual se cargó del CSV).

In [ ]:
columnas_metricas = [
    'Depth(m)', 'ROP (m/hr)', 'weight on bit (kg)', 'Temp Out( degC)',
    'Temp In(degC)', 'Pit Total (m3)', 'Pump Press (KPa)',
    'Hookload (kg)', 'Surface Torque (KPa)', 'Flow In(liters/min)',
    'WH Pressure (KPa)'
]

df_imperial = df.drop(columns=columnas_metricas).copy()
print('Columnas restantes:', df_imperial.shape[1])
df_imperial.columns

## 4. Correlación con el ROP

Queremos ver qué variables se relacionan más con el ROP. Usamos `df.corr()`, que calcula el coeficiente de correlación de Pearson: un número entre -1 y 1 que mide qué tan relacionadas están dos variables de forma lineal (cuanto más cerca de -1 o +1, más fuerte la relación; cerca de 0, casi no hay relación).

Con esta sola lista ya podemos decidir qué columnas sacar (sección 5) — recién después profundizamos con gráficos en las que quedaron (sección 6).

In [ ]:
correlaciones = df_imperial.corr(numeric_only=True)['ROP (ft/hr)'].sort_values(ascending=False)
correlaciones

## 5. Selección final de columnas

Con la lista de arriba, sacamos las columnas sin relación causal creíble con el ROP:
- `H2S Floor`, `H2S Cellar`, `H2S Pits` → detectores de gas tóxico, sin explicación física para afectar el ROP.
- `Flow Out %` → correlación casi nula (-0.12) y es variable de control/seguridad, no mecánica.

El resto se conserva, ya que sigue siendo información real del proceso de perforación, útil para un futuro modelo.

Trabajamos sobre otra copia nueva, `df_final`, sin tocar `df` ni `df_imperial`.

In [ ]:
df_final = df_imperial.drop(columns=['H2S Floor', 'H2S Cellar', 'H2S Pits', 'Flow Out %']).copy()
print('Columnas finales:', df_final.shape[1])
df_final.columns

## 6. Profundizando en las variables que quedaron

Ya decidimos qué se queda (`df_final`, 12 columnas). Ahora sí vale la pena mirar con más detalle solo estas variables: un mapa de calor para verlas todas juntas, una tabla con el significado físico de cada una, y gráficos de dispersión para confirmar visualmente la forma real de cada relación.

### 6.1 Mapa de calor

Mismo cálculo de correlación de Pearson, pero de todas las variables de `df_final` contra todas a la vez, para verlo de un vistazo.

In [ ]:
plt.figure(figsize=(11, 9))
sns.heatmap(df_final.corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Matriz de correlación completa (variables finales)')
plt.show()

### 6.2 Qué significa cada variable, y su relación con el ROP

| Variable | Qué es (significado físico) | Corr. vs ROP | Interpretación |
|---|---|---|---|
| `Depth (ft)` | Profundidad alcanzada | -0.51 | A mayor profundidad, la formación suele ser más compacta y el ROP baja. |
| `weight on bit (k-lbs)` | Peso aplicado sobre el trépano | -0.52 | La teoría de perforación dice que más peso debería *subir* el ROP (hasta cierto punto). Que acá dé inverso es una señal de que hay otros factores de fondo (por ejemplo, el tipo de roca) que no podemos aislar sin datos de litología. |
| `Rotary Speed (rpm)` | Velocidad de rotación de la sarta | +0.29 | Va en la dirección física esperada: más RPM, más ROP. |
| `Surface Torque (psi)` | Resistencia que encuentra la sarta al rotar | -0.38 | Consistente con roca más dura: más torque, menos ROP. |
| `Pump Press (psi)` | Presión de bombeo del lodo (limpieza e hidráulica) | -0.49 | Relación moderada; la presión de bombeo se ajusta según las condiciones del pozo. |
| `Flow In (gal/min)` | Caudal de lodo bombeado, ayuda a limpiar el pozo | +0.48 | Mejor limpieza del pozo puede favorecer el ROP. |
| `Hookload (k-lbs)` | Peso que cuelga del gancho (peso de la sarta sin apoyar en el fondo) | -0.43 | Relación moderada; el hookload también depende de cuánta tubería hay metida en el pozo. |
| `Temp Out (degF)` | Temperatura del lodo que retorna a superficie | -0.43 | Relación moderada. |
| `Temp In (degF)` | Temperatura del lodo que entra al pozo, en superficie | -0.22 | Relación débil — es de las variables que menos influyen. |
| `Pit Total (bbls)` | Volumen de lodo en los tanques de superficie | +0.235 | Relación débil a moderada. |
| `WH Pressure (psi)` | Presión en la cabeza del pozo | -0.50 | Relación moderada; más ligada a control de pozo que a la mecánica del trépano. |

### 6.3 Dispersión (scatter): ¿la relación es realmente una línea recta?

El número de correlación asume que la relación es una línea recta, pero eso no siempre es así. Estos gráficos nos permiten ver con los ojos la forma real de la relación entre cada variable y el ROP, en vez de confiar solo en el número.

In [ ]:
variables_top = ['weight on bit (k-lbs)', 'Depth(ft)', 'WH Pressure (psi)',
                 'Pump Press (psi)', 'Flow In (gal/min)', 'Hookload (k-lbs)']

variables_predictoras = ['weight on bit (k-lbs)', 'Depth(ft)', 'WH Pressure (psi)',
                          'Pump Press (psi)', 'Flow In (gal/min)', 'Hookload (k-lbs)',
                          'Rotary Speed (rpm)', 'Surface Torque (psi)',
                          'Temp Out( degF)', 'Temp In(degF)', 'Pit Total (bbls)']

fig, axes = plt.subplots(3, 4, figsize=(18, 11))
for ax, col in zip(axes.flatten(), variables_predictoras):
    sns.scatterplot(x=df_final[col], y=df_final['ROP (ft/hr)'], ax=ax, alpha=0.3)
    ax.set_title(f'{col} vs ROP')
plt.tight_layout()
plt.show()

**Qué se ve acá:** en casi todos los gráficos aparece el mismo patrón general — una nube densa de puntos concentrada en un rango de valores (con el ROP variando mucho ahí, desde 0 hasta valores altos), y después una "cola" de puntos que se extiende hacia otro rango donde el ROP se aplana cerca de 0. No es una línea recta clara en ningún caso — es consistente con que ninguna correlación individual se acerca a ±1 (todas están entre 0.2 y 0.5 en valor absoluto): hay relación real, pero con mucho ruido y dispersión, no una relación limpia y directa.

Un detalle que llama la atención: `WH Pressure (psi)` tiene valores negativos (de -1200 a 0), lo cual es raro para una presión — probablemente es un criterio de signo del sensor/software de registro, no necesariamente un error, pero vale la pena tenerlo en cuenta si se usa esta variable más adelante.

### 6.4 Conclusión

No contamos con una variable de litología (tipo de roca) en este dataset. Con litología podríamos separar el análisis por tipo de formación y entender mejor la causa real de cada relación; sin ella, estas correlaciones son un buen punto de partida pero tienen esa limitación conocida. Por eso conseguir litología queda anotado como prioridad en los próximos pasos.

## 7. Distribución de cada variable (histogramas)

Antes de contar outliers, miramos la **forma** de cada variable. Esto importa porque el método de detección de outliers que tiene sentido usar depende de si la distribución es más o menos simétrica (parecida a una campana) o si tiene una cola larga hacia un lado (sesgada).

In [ ]:
numeric_cols = df_final.select_dtypes(include=np.number).columns

fig, axes = plt.subplots(3, 4, figsize=(18, 11))
for ax, col in zip(axes.flatten(), numeric_cols):
    sns.histplot(df_final[col], kde=True, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

**Cómo leer esto:** si una variable se ve como una campana más o menos simétrica, el método IQR va a dar un número de outliers razonable. Si se ve con una cola larga hacia un lado (por ejemplo, muchos valores bajos y pocos picos altos), el método IQR va a marcar buena parte de esa cola como "outlier" aunque sea el comportamiento normal de la variable — hay que tenerlo en cuenta al interpretar la sección siguiente.

## 8. Outliers

### 8.1 Conteo (IQR y z-score, para comparar)

Contamos con los dos métodos vistos en el curso. Si los dos coinciden en un número parecido, la variable es razonablemente simétrica. Si difieren mucho (como va a pasar acá), es la señal de sesgo que vimos en los histogramas de arriba — no hay que tomar ninguno de los dos números como verdad absoluta sin mirar el histograma.

In [ ]:
def contar_outliers(serie):
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    return ((serie < limite_inferior) | (serie > limite_superior)).sum()

for col in numeric_cols:
    print(col, '->', contar_outliers(df_final[col]), 'outliers (IQR)')

In [ ]:
for col in numeric_cols:
    z = np.abs(stats.zscore(df_final[col]))
    print(col, '->', (z > 3).sum(), 'outliers (z-score)')

**IQR vs z-score, ¿cuál conviene para cada variable?**

El z-score se calcula con la media y el desvío estándar — funciona bien cuando la variable tiene forma de campana (simétrica). El IQR se calcula con cuartiles — no le afectan tanto los valores extremos, por eso es más confiable en variables con cola larga (sesgadas).

Mirando los histogramas de la sección 7: `ROP`, `WH Pressure`, `Surface Torque` y `weight on bit` NO tienen forma de campana (están sesgadas, con cola larga hacia un lado) — para esas, conviene guiarse por IQR, y tomar el número de z-score con desconfianza (probablemente esté subestimando outliers reales, porque los valores extremos ya "estiraron" el desvío estándar usado para calcularlo). `Depth`, `Temp Out`, `Temp In` y `Pit Total` se ven más simétricas — ahí los dos métodos deberían dar números más parecidos entre sí, y podés confiar en cualquiera de los dos.

Importante: que IQR marque muchos outliers en una variable sesgada **no significa que esas filas estén mal** — puede ser simplemente la forma natural de esa variable (como vimos en la nota de la sección 7). Por eso no alcanza con el número solo — hace falta la investigación caso por caso que hicimos en 8.3, 8.4 y 8.5.

### 8.2 Boxplots (apoyo visual a los outliers)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 11))
for ax, col in zip(axes.flatten(), numeric_cols):
    sns.boxplot(x=df_final[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

### 8.3 Empezamos por la variable principal: ROP

En el scatter de la sección 6 vimos una fila con un ROP extremo (cerca de 3000 ft/hr). Antes de mirar las demás variables, investigamos esa fila puntual.

In [ ]:
fila_maxima = df_final.loc[df_final['ROP (ft/hr)'].idxmax()]
fila_maxima

Esa fila tiene `weight on bit = 0.00` — peso cero sobre el trépano. Con el trépano sin apoyo real, ese ROP no es perforación genuina. Antes de asumir que es un caso único, contamos cuántas filas del dataset tienen esa misma condición.

In [ ]:
filas_wob_cero = df_final[df_final['weight on bit (k-lbs)'] == 0]
print('Filas con weight on bit = 0:', filas_wob_cero.shape[0])
filas_wob_cero[['Depth(ft)', 'ROP (ft/hr)', 'Rotary Speed (rpm)']].describe()

**198 filas** tienen WOB=0 — no es un caso aislado. Esas filas convendría excluirlas antes de entrenar un modelo, porque no reflejan una relación real entre los parámetros y la velocidad de perforación.

**Chequeo de consistencia: si no hay perforación real, ¿por qué el ROP no es siempre bajo?**

Buena pregunta para hacerse antes de seguir: si la sarta estuviera simplemente girando en el mismo lugar sin avanzar, el ROP debería dar bajo, no alto. Miramos cómo se distribuye el ROP dentro de las 198 filas de WOB=0, para confirmar qué pasa en cada caso.

In [ ]:
print(filas_wob_cero['ROP (ft/hr)'].describe())
print()
print('ROP < 50 (posible "girando sin avanzar"):', (filas_wob_cero['ROP (ft/hr)'] < 50).sum())
print('ROP > 500 (posible maniobra, sarta moviéndose rápido):', (filas_wob_cero['ROP (ft/hr)'] > 500).sum())

**Resultado:** de las 198 filas, solo **6** tienen ROP bajo (<50 ft/hr) y **36** tienen ROP muy alto (>500 ft/hr); el resto está en un rango intermedio. Antes de sacar conclusiones, chequeamos el salto de profundidad entre filas consecutivas en cada grupo — porque si el ROP alto fuera por "la sarta moviéndose más", el salto debería ser mayor ahí.

In [ ]:
df_final['salto_profundidad'] = df_final['Depth(ft)'].diff()

wob_cero_bajo = filas_wob_cero[filas_wob_cero['ROP (ft/hr)'] < 50].index
wob_cero_alto = filas_wob_cero[filas_wob_cero['ROP (ft/hr)'] > 500].index

print('Salto de profundidad, WOB=0 y ROP bajo:')
print(df_final.loc[wob_cero_bajo, 'salto_profundidad'].describe())
print()
print('Salto de profundidad, WOB=0 y ROP alto:')
print(df_final.loc[wob_cero_alto, 'salto_profundidad'].describe())

**El salto de profundidad es prácticamente el mismo en los dos grupos (~1 pie por fila)** — no es que la profundidad "salte más" cuando el ROP es alto. Lo que cambia es el **tiempo** que tardaron en avanzar ese mismo pie, no la distancia.

Con esto, la explicación correcta es: en las dos situaciones **sí se está profundizando de verdad** (a diferencia de lo que dijimos antes). La diferencia es el tiempo:
- **ROP alto con WOB=0:** ese pie se recorrió en muy poco tiempo — consistente con estar bajando la herramienta a través de un tramo de pozo **ya perforado antes** (hueco abierto, sin roca nueva que cortar, así que no hace falta peso y se avanza rápido).
- **ROP bajo con WOB=0:** tardaron mucho en ese mismo pie — consistente con estar detenidos ahí (por ejemplo, una conexión de tubería).

En los dos casos las 198 filas siguen sin ser perforación real — pero no por "sacar herramienta" (eso ni tendría sentido con la profundidad subiendo), sino por estar pasando por hueco ya abierto o parados, sin cortar roca nueva.

### 8.4 ¿Alcanza con mirar solo WOB? Sumamos Rotary Speed (RPM)

WOB=0 es una señal confiable de "no hay perforación real". Pero hay un matiz con el RPM: en perforación direccional existe una técnica llamada **sliding** (deslizamiento), donde se perfora **con un motor de fondo sin rotar la sarta desde superficie** — ahí RPM puede ser 0 y aun así **sí** se está perforando. Por eso RPM=0 solo no es una señal tan limpia como WOB=0.

Cruzamos las dos variables sobre los 978 outliers de ROP, en tres grupos:
- **WOB=0** → no hay apoyo real, artefacto confirmado. (artefacto:término de análisis de datos que representa un valor que aparece en los datos, pero que no representa el fenómeno real que se quiere medir)
- **WOB>0 y RPM=0** → podría ser *sliding* real, o podría ser otra maniobra — ambiguo, no lo descartamos sin más evidencia.
- **WOB>0 y RPM>0** → perforación normal en curso — el candidato más creíble a evento real (formación blanda), no artefacto.

In [ ]:
rop_outliers_altos = df_final['ROP (ft/hr)'] > (df_final['ROP (ft/hr)'].quantile(0.75) + 1.5*(df_final['ROP (ft/hr)'].quantile(0.75)-df_final['ROP (ft/hr)'].quantile(0.25)))
wob_cero = df_final['weight on bit (k-lbs)'] == 0
rpm_cero = df_final['Rotary Speed (rpm)'] == 0

print('Outliers de ROP con WOB=0 (artefacto confirmado):', (rop_outliers_altos & wob_cero).sum())
print('Outliers de ROP con WOB>0 y RPM=0 (ambiguo, posible sliding):', (rop_outliers_altos & ~wob_cero & rpm_cero).sum())
print('Outliers de ROP con WOB>0 y RPM>0 (evento real probable):', (rop_outliers_altos & ~wob_cero & ~rpm_cero).sum())

**Resultado: 166 artefacto confirmado, 42 ambiguos (posible sliding), 770 evento real probable.** La gran mayoría de los outliers de ROP (770 de 978) tiene WOB y RPM normales — son probablemente tramos de perforación genuinamente rápida, no errores. Solo las 166 con WOB=0 tienen justificación sólida para eliminarse; las 42 ambiguas y las 770 restantes se quedan, por ahora, sin tocar.

### 8.5 ¿Esto también explica los outliers de Surface Torque?

`Surface Torque` tuvo 930 outliers en el conteo de 8.1 — un número parecido al de ROP. Chequeamos cuántos de esos coinciden con las 198 filas de WOB=0 ya confirmadas.

In [ ]:
def es_outlier(serie):
    Q1 = serie.quantile(0.25)
    Q3 = serie.quantile(0.75)
    IQR = Q3 - Q1
    return (serie < Q1 - 1.5 * IQR) | (serie > Q3 + 1.5 * IQR)

li_torque = df_final['Surface Torque (psi)'].quantile(0.25) - 1.5*(df_final['Surface Torque (psi)'].quantile(0.75)-df_final['Surface Torque (psi)'].quantile(0.25))
ls_torque = df_final['Surface Torque (psi)'].quantile(0.75) + 1.5*(df_final['Surface Torque (psi)'].quantile(0.75)-df_final['Surface Torque (psi)'].quantile(0.25))
torque_outliers = es_outlier(df_final['Surface Torque (psi)'])

print('Outliers bajos:', (df_final['Surface Torque (psi)'] < li_torque).sum())
print('Outliers altos:', (df_final['Surface Torque (psi)'] > ls_torque).sum())
print()
coincidencia = (torque_outliers & wob_cero).sum()
print('Surface Torque:', torque_outliers.sum(), 'outliers, de los cuales', coincidencia, 'tienen WOB=0', f'({coincidencia/torque_outliers.sum()*100:.0f}%)')
print()
print('Torque promedio cuando WOB=0:', df_final.loc[wob_cero, 'Surface Torque (psi)'].mean())
print('Torque promedio cuando WOB>0:', df_final.loc[~wob_cero, 'Surface Torque (psi)'].mean())

**A diferencia del ROP, acá la mayoría de los outliers de Torque (722 de 930) son bajos, no altos** — y tiene una explicación física directa y coherente: sin peso aplicado sobre el trépano, la mecha casi no encuentra resistencia al girar, así que el torque necesario cae mucho (de ~134 psi en promedio a ~12 psi). No es un artefacto de cálculo como el ROP — es la física esperada. Aun así, el WOB=0 solo explica una parte de los 930 outliers; el resto queda con la misma causa sin identificar que las 770 filas de ROP. No investigamos más allá con lo que tenemos: haría falta litología o el reporte diario del pozo.

### 8.6 Decisión final: ¿cuántas filas eliminamos, y cómo cambia la data?

Con toda la evidencia junta, la única eliminación con fundamento sólido son las **198 filas con weight on bit = 0**.

**Aclaración importante para no confundir los números:** en 8.4 vimos que, de los 978 outliers de ROP, **166** también tenían WOB=0 — ese 166 fue solo evidencia para justificar la decisión (mostraba que la mayoría de los outliers de ROP coincidían con WOB=0). No es lo que se elimina. **Lo que se elimina son las 198**, es decir, TODAS las filas con WOB=0 del dataset completo — incluyendo las 32 que no llegaron a ser outlier de ROP (tenían un valor "normal"), porque el criterio de eliminación es "no hay perforación real" (WOB=0), no "ser un outlier estadístico".

Armamos `df_limpio` sin esas 198 filas, y comparamos la estadística general antes y después.

In [ ]:
print('ANTES (df_final):')
display(df_final[numeric_cols].describe())

In [ ]:
df_limpio = df_final[df_final['weight on bit (k-lbs)'] > 0].copy()
print('Filas antes:', df_final.shape[0], '| Filas después:', df_limpio.shape[0])
print()
print('DESPUÉS (df_limpio, sin WOB=0):')
display(df_limpio[numeric_cols].describe())

Sobre todo la fila `min` y `mean` del ROP y del Surface Torque — deberían subir un poco al sacar las filas de peso cero, ya que esas filas metían valores extremos y ceros artificiales en el promedio.

**Un último vistazo visual, ya con la data limpia de ROP respeto a RPM y Surface Torque:**

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, col in zip(axes.flatten(), variables_top):
    sns.scatterplot(x=df_limpio[col], y=df_limpio['ROP (ft/hr)'], ax=ax, alpha=0.3)
    ax.set_title(f'{col} vs ROP (sin WOB=0)')
plt.tight_layout()
plt.show()

### 8.7 Profundizando en otras variables

#### WH Pressure — corrección de la hipótesis anterior

La primera hipótesis ("cambio de calibración al inicio del pozo") **no se sostuvo** al revisar el dato completo: los valores negativos no están solo al principio, aparecen repartidos en casi todo el rango de profundidad (desde 85 ft hasta 7325 ft).

In [ ]:
print(df['WH Pressure (psi)'].describe())
print()
print('Valores negativos:', (df['WH Pressure (psi)'] < 0).sum(), 'de', len(df))

El grueso de los datos (percentil 25 a 75: entre 2.92 y 8.26 psi) son valores chicos y positivos — coherente con lo esperado en perforación normal (pozo abierto, circulando, sin cierre de BOP: la presión en cabeza no depende de la profundidad del BHA ni de la hidrostática de la columna, solo de la contrapresión de superficie). El problema son las **742 filas (10%) con valores negativos**, hasta -1231.83 psi — una presión absoluta negativa no es físicamente posible en un sistema abierto a la atmósfera.

**Conclusión:** esta situación apunta a un **problema de sensor o calibración intermitente**, no un fenómeno real ligado a ninguna variable mecánica. Se marca como variable con calidad de dato cuestionable — no se puede "explicar" cruzándola con otra columna.

**Qué hacer:**  marcar los valores negativos como `NaN` (son físicamente imposibles) e imputarlos — con la media/mediana (lo visto en el curso) o, mejor, con interpolación (`.interpolate()`), ya que es una serie ordenada por profundidad. Lo hacemos ahora mismo, antes de seguir con las demás variables.

In [ ]:
negativos_antes = (df_limpio['WH Pressure (psi)'] < 0).sum()
df_limpio['WH Pressure (psi)'] = df_limpio['WH Pressure (psi)'].where(df_limpio['WH Pressure (psi)'] >= 0)
print('Valores marcados como NaN (eran negativos):', negativos_antes)

df_limpio['WH Pressure (psi)'] = df_limpio['WH Pressure (psi)'].interpolate(limit_direction='both')
print('Nulos restantes después de interpolar:', df_limpio['WH Pressure (psi)'].isna().sum())

**Nota:** la primera vez que corrí esto, quedaron 60 valores sin rellenar — `.interpolate()` por defecto no puede completar un NaN si está al principio o al final de la serie (no tiene un vecino válido de ese lado para calcular el promedio). Agregar `limit_direction='both'` le dice que, en esos bordes, use el valor válido más cercano en vez de dejarlo vacío.

In [ ]:
df_limpio['WH Pressure (psi)'].describe()

El mínimo ya no debería ser negativo — confirmá que dé un valor chico y positivo, coherente con el resto de la columna.

#### Flow In — un fenómeno distinto al de WOB=0

De los 113 outliers, la mayoría (108) son **bajos**, no altos — y la mediana ahí es literalmente **0** (bombas apagadas). Solo 5 son altos (hasta 3317 gal/min, probablemente una maniobra puntual de limpieza de pozo).

In [ ]:
flow_bajo = df_final['Flow In (gal/min)'] < 313.7
wob_cero_mask = df_final['weight on bit (k-lbs)'] == 0

print('Filas con Flow In bajo:', flow_bajo.sum())
print('De esas, cuántas también tienen WOB=0:', (flow_bajo & wob_cero_mask).sum())

**Solo 3 de esas 108 filas coinciden con WOB=0** — no es el mismo fenómeno que ya resolvimos. Hay ~105 filas donde las bombas estaban apagadas pero el trépano si tenía peso aplicado — algo inusual en perforación normal (generalmente se levanta el trépano antes de parar de circular). No hay una explicación operativa clara con las columnas disponibles; queda como otra variable con causa sin confirmar.

#### Rotary Speed — sí tiene explicación, y es un evento real

Los 15 outliers son todos altos (136 a 271 RPM).

In [ ]:
def bounds(s):
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - 1.5*IQR, Q3 + 1.5*IQR

li, ls = bounds(df_final['Rotary Speed (rpm)'])
mask_rpm = df_final['Rotary Speed (rpm)'] > ls
cols = ['Depth(ft)', 'ROP (ft/hr)', 'weight on bit (k-lbs)', 'Surface Torque (psi)']

print('Rango de profundidad:', df_final.loc[mask_rpm, 'Depth(ft)'].min(), '-', df_final.loc[mask_rpm, 'Depth(ft)'].max())
pd.DataFrame({'promedio_en_outliers': df_final.loc[mask_rpm, cols].mean(), 'promedio_general': df_final[cols].mean()})

Los 15 casos están **todos en el tramo más superficial del pozo** (85 a 1889 ft), justo donde el ROP es altísimo (232 vs 42 de promedio general) y el WOB/Torque son bajos. Es la firma típica de la **sección superior del pozo** (formación blanda cerca de superficie): se rota rápido, con poco peso, y avanza fácil porque la roca casi no resiste.

**Conclusión: evento real, no un artefacto.** No hace falta eliminar estas filas.

#### Pit Total — probablemente ligado al mismo tramo rápido

Los 113 outliers son todos bajos.

In [ ]:
li2, ls2 = bounds(df_final['Pit Total (bbls)'])
mask_pit = df_final['Pit Total (bbls)'] < li2
cols2 = ['Depth(ft)', 'ROP (ft/hr)', 'Flow In (gal/min)']

print('Rango de profundidad:', df_final.loc[mask_pit, 'Depth(ft)'].min(), '-', df_final.loc[mask_pit, 'Depth(ft)'].max())
pd.DataFrame({'promedio_en_outliers': df_final.loc[mask_pit, cols2].mean(), 'promedio_general': df_final[cols2].mean()})

El ROP en estas filas también es más alto que el promedio (129 vs 42) y el caudal es mayor (942 vs 716 gal/min) — consistente con el mismo tramo de perforación rápida que vimos en Rotary Speed, con más actividad de circulación bajando el nivel de los tanques. No es un caso tan limpio como Rotary Speed, pero tampoco contradice la misma historia — probablemente real, no un error.

### 8.8 Resumen final

Con todo ya analizado y corregido variable por variable, así queda el panorama completo de la sección 8.

**Resumen final: qué se hizo con cada variable**

| Variable | Outliers (IQR) | Qué se hizo |
|---|---|---|
| `weight on bit` | 209 | ✅ Eliminadas 198 filas con WOB=0 |
| `ROP` | 978 | ✅ 166 eliminadas (mismas filas de WOB=0) — 42 ambiguas y 770 reales se conservan, sin tocar |
| `Surface Torque` | 930 | ✅ La mayoría eliminada de rebote (mismas filas de WOB=0) — el resto se conserva |
| `WH Pressure` | 251 | ✅ Valores negativos reemplazados por interpolación |
| `Flow In` | 113 | Se conserva, sin acción — sin explicación clara |
| `Rotary Speed` | 15 | Se conserva — evento real (sección superficial del pozo) |
| `Pit Total` | 113 | Se conserva — probablemente ligado al mismo evento real |

`df_limpio` queda como la versión final: sin las filas de WOB=0, y con `WH Pressure` corregida. De acá en adelante, todo el análisis sigue con `df_limpio`, no con `df_final`.

In [ ]:
print('Filas en df_final:', df_final.shape[0])
print('Filas en df_limpio:', df_limpio.shape[0])
df_limpio.describe()

---

**Con esto cerramos el análisis de outliers**, con todas las decisiones ya aplicadas en `df_limpio` (8.8). Lo que sigue (pairplot, estadística, escalado, clustering) usa `df_limpio` de acá en adelante, no `df_final`.

---

## 9. Pairplot de las variables principales (cierre visual)

Un pairplot es una grilla que combina todos los scatter y todas las distribuciones de un grupo de variables en una sola imagen — la versión "todo junto" del scatter que ya hicimos en la sección 6.

In [ ]:
sns.pairplot(df_limpio[variables_top + ['ROP (ft/hr)']])
plt.show()

**Cómo leer esto:** cada celda de la grilla es un scatter entre dos variables (fila y columna), y la diagonal muestra la distribución de cada variable sola (como los histogramas de la sección 7, pero recalculados sobre `df_limpio`). Es la misma idea que el scatter de la sección 6, pero ahora mostrando **todas las combinaciones entre las variables principales a la vez**, no solo cada una contra el ROP — sirve para detectar si dos variables predictoras están relacionadas fuertemente entre sí (por ejemplo, si WOB y Depth se mueven juntas), información que un modelo después podría aprovechar o que podría ser redundante.

## 10. Estadística descriptiva y valores extremos

 Sirve como un chequeo final de sanidad sobre `df_limpio`: confirma que ya no hay valores imposibles (como el WH Pressure negativo o el ROP de 2977) y da una foto general de en qué rango se mueve cada variable, antes de pasar a un modelo.

In [ ]:
df_limpio.describe()

In [ ]:
fila_rop_max = df_limpio['ROP (ft/hr)'].idxmax()
fila_rop_min = df_limpio['ROP (ft/hr)'].idxmin()

print('Profundidad con ROP máximo:', df_limpio.loc[fila_rop_max, 'Depth(ft)'], 'ft, ROP =', df_limpio['ROP (ft/hr)'].max())
print('Profundidad con ROP mínimo:', df_limpio.loc[fila_rop_min, 'Depth(ft)'], 'ft, ROP =', df_limpio['ROP (ft/hr)'].min())

Estas dos filas puntuales (la de ROP más alto y más bajo dentro de `df_limpio`, ya sin las filas de WOB=0) sirven como un último chequeo de sanidad: confirman que, incluso en el dataset limpio, los extremos siguen teniendo sentido físico — no deberían aparecer valores absurdos si la limpieza funcionó bien.

## 11. Escalado (preparación para un futuro modelo)

**Importante:** esto NO sirve para saber qué variable influye más — la correlación no cambia si escalamos los datos, es matemáticamente invariante a la escala. El escalado solo pone a todas las variables en un rango comparable, y hace falta recién si más adelante entrenás un modelo sensible a la escala (KNN, SVM, regresión regularizada, redes neuronales). Para árboles/Random Forest no hace falta.

Usamos `RobustScaler` (no `StandardScaler` ni `MinMaxScaler`) porque, como vimos en la Notebook 12, es el que corresponde cuando hay outliers reales en los datos — usa la mediana y el rango intercuartílico en vez de la media y el desvío estándar, así los outliers no distorsionan la escala del resto de los datos.

In [ ]:
columnas_a_escalar = [c for c in numeric_cols]

scaler = RobustScaler()
datos_escalados = scaler.fit_transform(df_limpio[columnas_a_escalar])
df_limpio_scaled = pd.DataFrame(datos_escalados, columns=columnas_a_escalar)

print('Antes de escalar:')
display(df_limpio[columnas_a_escalar].describe().loc[['min', 'max', 'mean']])
print('Después de escalar (RobustScaler):')
display(df_limpio_scaled.describe().loc[['min', 'max', 'mean']])

## 12. Agrupando filas parecidas (clustering)

Ya vimos que no tenemos litología, y que dividir el pozo en tramos de profundidad iguales (lo que hacíamos antes en esta sección) no aportaba mucho. En vez de cortar por profundidad a ciegas, probamos algo distinto: **K-Means**, un método que agrupa filas según qué tan parecidos son sus valores en varias variables a la vez (WOB, RPM, Torque, etc.), sin mirar la profundidad directamente. Si esos grupos terminan relacionándose con el ROP o con la profundidad, es una señal de que capturaron algo real (como un cambio de formación), sin que se lo hayamos dicho nosotras.

**Por qué hace falta el escalado acá:** K-Means calcula distancias entre filas usando todas las variables a la vez. Si no estuvieran en la misma escala, una variable con números grandes (como Pump Press, hasta 2200) pesaría mucho más que una con números chicos (como WH Pressure, hasta 17) sin que eso represente su importancia real. Por eso usamos `df_limpio_scaled`, que ya armamos en la sección anterior — no quedó sin usar, era para este paso.

Elegimos 3 grupos (`n_clusters=3`) para arrancar — es una elección arbitraria mía, igual que el `bins=10` de antes; no hay una razón profunda para elegir 3 en vez de 2 o 4, es un punto de partida para ver si aparece algo interesante.

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_limpio['cluster'] = kmeans.fit_predict(df_limpio_scaled[variables_predictoras])

df_limpio.groupby('cluster')['ROP (ft/hr)'].agg(['mean', 'count'])

Comparamos también la profundidad de cada grupo, para ver si el clustering terminó relacionándose con la profundidad aunque no se la dimos como variable de referencia principal.

In [ ]:
df_limpio.groupby('cluster')['Depth(ft)'].agg(['mean', 'min', 'max'])

**Resultado:** los 3 grupos tienen un ROP promedio bien distinto entre sí (aproximadamente 12, 32 y 86 ft/hr), y el grupo de ROP más alto tiene, en promedio, menor profundidad — coherente con la zona superficial blanda que ya veníamos encontrando. A diferencia de los tramos por profundidad, acá **los rangos de profundidad de cada grupo se superponen bastante** (por ejemplo, el grupo de ROP alto va de 91 a 6577 ft) — esto tiene sentido, porque el agrupamiento no corta por profundidad directamente, sino por el comportamiento conjunto de varias variables mecánicas; puede haber momentos a mayor profundidad que "se comportan" parecido a la zona superficial, y eso es justamente lo interesante de este método frente a cortar a ciegas.

No es una prueba definitiva de litología (para eso seguimos necesitando el dato real), pero es una segunda forma, más rica que los tramos fijos, de aproximar dónde cambia el comportamiento del pozo.

## 13. Próximos pasos

- Conseguir litología (tipo de roca por tramo) para poder separar el efecto real de cada parámetro mecánico del efecto de la profundidad, y recién ahí sí aplicar un encoder sobre esa columna categórica.
- Evaluar más pozos (si se consiguen) para no depender del comportamiento de uno solo.
- Investigar la causa de las ~105 filas de `Flow In` en cero con WOB>0 (8.7) y de la mayoría de los outliers de ROP y Surface Torque (770 filas, 8.4) que no son artefactos confirmados — no hay evidencia suficiente para descartarlas, pero tampoco para confirmar que son eventos reales sin litología o el reporte diario del pozo.
- Recién al momento de entrenar el modelo predictivo: `train_test_split`, y usar `df_limpio_scaled` solo si el modelo elegido lo necesita.